In [ ]:
"""
Comprehensive Multi File Strategy Analyzer
Aligned with Miroslav metric triad

Metric definitions

1 Authority Yield
  Mean relative PageRank change per inserted internal link
  Formula: mean_delta_pct divided by k

2 Authority Volatility
  Standard deviation of Authority Yield across simulation runs
  Source: read std_delta from CSV and normalize by k per inserted link
  If std_delta is missing, Authority Volatility is undefined and reported as NaN

3 Down Up Ratio
  Expected pages_down divided by expected pages_up
  Formula: avg_pages_down divided by max(avg_pages_up, 1)

Notes
  Authority Volatility is reported only when run level dispersion statistics are available
  Volatility is not inferred from confidence intervals or aggregated estimates
  Strategy ordering and colors are fixed and reused across all plots using tab10
  Naming Convention: all strategies use the format "[Name] candidates"
"""

from dataclasses import dataclass
from typing import List, Tuple, Optional
from enum import Enum
import os
import re
from datetime import datetime
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

try:
    from google.colab import drive

    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False
    drive = None

warnings.filterwarnings("ignore")


# ============================================================
# Configuration and constants
# ============================================================

STRATEGY_ORDER = [
    "low candidates",
    "high candidates",
    "mixed candidates",
    "random candidates",
    "folder candidates",
]

TYPE_ORDER = ["automatic", "expert"]

TAB10 = mpl.colormaps["tab10"]
STRATEGY_COLORS = {s: TAB10(i % 10) for i, s in enumerate(STRATEGY_ORDER)}
TYPE_COLORS = {"automatic": TAB10(6), "expert": TAB10(7)}
DISTRIBUTION_COLORS = {"up": TAB10(2), "down": TAB10(3), "neutral": TAB10(8)}

MEAN_COL_CANDIDATES = ["mean_delta_pct", "delta_pr_percent_mean"]
STD_COL_CANDIDATES = ["std_delta", "delta_pr_percent_std"]


class Config:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.initialize()
        return cls._instance

    def initialize(self):
        self.TOTAL_KALICUBE_PAGES = 1841
        self.INSERTED_LINKS_K = 240
        self.BASE_PATH = "/content/drive/MyDrive/WebKnoGraph"
        self.OUTPUT_PATH = "/content/drive/MyDrive/WebKnoGraph"
        self.DPI = 300
        self.STYLE = "seaborn-v0_8-whitegrid"


class FileType(Enum):
    AUTOMATIC = "automatic"
    EXPERT = "expert"
    OTHER = "other"


class SubCategory(Enum):
    BA = "ba"
    REAL_WWW = "real_www"
    UNKNOWN = "unknown"


@dataclass
class FileInfo:
    full_path: str
    relative_path: str
    filename: str
    folder: str
    size_kb: float
    modified: str
    type: FileType
    sub_category: SubCategory


# ============================================================
# Helper utilities
# ============================================================


def normalize_range_to_label(r: str) -> str:
    """
    Converts values like "5_35" or "5-35" or "5–35" into a readable label: "5 to 35".
    """
    if r is None or (isinstance(r, float) and np.isnan(r)):
        return "Unknown"
    s = str(r).strip()
    s = s.replace("–", " ").replace("--", " ").replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    nums = [int(x) for x in re.findall(r"\d+", s)]
    if len(nums) >= 2:
        return f"{nums[0]} to {nums[1]}"
    return s if s else "Unknown"


def range_sort_key(label: str) -> Tuple[int, int, str]:
    """
    Sorts '5 to 35' before '35 to 65' etc, keeps Unknown last.
    """
    if not isinstance(label, str):
        return (10**9, 10**9, "")
    if label.strip().lower() == "unknown":
        return (10**9, 10**9, "unknown")
    nums = [int(x) for x in re.findall(r"\d+", label)]
    if len(nums) >= 2:
        return (nums[0], nums[1], label)
    if len(nums) == 1:
        return (nums[0], nums[0], label)
    return (10**9 - 1, 10**9 - 1, label)


def ensure_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def pick_first_existing_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    mask = (~v.isna()) & (~w.isna()) & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.average(v[mask], weights=w[mask]))


def group_weighted_mean(
    df: pd.DataFrame, value_col: str, weight_col: str = "total_simulations"
) -> float:
    if value_col not in df.columns:
        return np.nan
    if weight_col not in df.columns:
        return float(pd.to_numeric(df[value_col], errors="coerce").mean())
    return weighted_mean(df[value_col], df[weight_col])


# ============================================================
# File management
# ============================================================


class FileManager:
    def __init__(self):
        self.config = Config()
        self.mount_drive()

    def mount_drive(self):
        if COLAB_AVAILABLE:
            if not os.path.exists("/content/drive"):
                print("Mounting Google Drive")
                drive.mount("/content/drive", force_remount=True)
                print("Google Drive mounted\n")
            else:
                print("Google Drive already mounted\n")
        else:
            print("Google Colab not detected, skipping Drive mount")
            print(f"Ensure files are accessible at {self.config.BASE_PATH}\n")

    def categorize_file(self, file_path: str) -> Tuple[FileType, SubCategory]:
        path_lower = file_path.lower()

        if "automatic" in path_lower:
            type_category = FileType.AUTOMATIC
        elif "expert" in path_lower or "expert_led" in path_lower:
            type_category = FileType.EXPERT
        else:
            type_category = FileType.OTHER

        if "ba_" in path_lower or "/ba" in path_lower or "ba_results" in path_lower:
            sub_category = SubCategory.BA
        elif "real_www" in path_lower or "realwww" in path_lower:
            sub_category = SubCategory.REAL_WWW
        else:
            sub_category = SubCategory.UNKNOWN

        return type_category, sub_category

    def find_multi_files(self) -> List[FileInfo]:
        all_files: List[FileInfo] = []

        print("Searching for multi csv files")
        print("=" * 60)

        if not os.path.exists(self.config.BASE_PATH):
            print(f"Base path does not exist: {self.config.BASE_PATH}")
            return []

        for root, dirs, files in os.walk(self.config.BASE_PATH):
            dirs[:] = [d for d in dirs if not d.startswith(".") and "Trash" not in d]

            for file in files:
                if file.lower().startswith("multi") and file.lower().endswith(".csv"):
                    full_path = os.path.join(root, file)
                    relative_path = full_path.replace(self.config.BASE_PATH + "/", "")

                    try:
                        file_size = os.path.getsize(full_path)
                        mod_time = os.path.getmtime(full_path)
                        mod_date = datetime.fromtimestamp(mod_time).strftime(
                            "%Y-%m-%d %H:%M"
                        )

                        type_cat, sub_cat = self.categorize_file(relative_path)

                        all_files.append(
                            FileInfo(
                                full_path=full_path,
                                relative_path=relative_path,
                                filename=file,
                                folder=os.path.basename(os.path.dirname(full_path)),
                                size_kb=file_size / 1024,
                                modified=mod_date,
                                type=type_cat,
                                sub_category=sub_cat,
                            )
                        )
                    except Exception as e:
                        print(f"Error accessing file {file}: {e}")

        print(f"Found {len(all_files)} files\n")
        return all_files


# ============================================================
# Visualization
# ============================================================


class UnifiedVisualizer:
    def __init__(self):
        self.config = Config()
        self.figures: List[plt.Figure] = []
        try:
            plt.style.use(self.config.STYLE)
        except OSError:
            plt.style.use("ggplot")
        self._apply_global_style()

    def _apply_global_style(self):
        mpl.rcParams.update(
            {
                "figure.dpi": self.config.DPI,
                "savefig.dpi": self.config.DPI,
                "font.size": 16,
                "axes.labelsize": 18,
                "axes.titlesize": 16,
                "xtick.labelsize": 14,
                "ytick.labelsize": 14,
                "legend.fontsize": 14,
            }
        )
        sns.set_context("talk", font_scale=1.0)
        sns.set_style("whitegrid")

    def create_all_plots(self, df: pd.DataFrame):
        if df.empty:
            print("DataFrame is empty, skipping plots")
            return

        print("\nGenerating plots using Miroslav metric triad")
        print("=" * 60)

        self._create_strategy_performance_plots(df)
        self._create_comparative_plots(df)

        print(f"Created {len(self.figures)} figures")
        for i, fig in enumerate(self.figures, 1):
            print(f"Displaying figure {i}")
            plt.show()

    def _create_strategy_performance_plots(self, df: pd.DataFrame):
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))
        # No suptitle and no panel titles, per professor feedback

        df = df.copy()
        if "total_simulations" not in df.columns:
            df["total_simulations"] = 1.0

        summary = (
            df.groupby("strategy", observed=True)
            .apply(
                lambda g: pd.Series(
                    {
                        "yield_mean": weighted_mean(
                            g["authority_yield"], g["total_simulations"]
                        ),
                        "volatility_mean": weighted_mean(
                            g["authority_volatility"], g["total_simulations"]
                        ),
                        "avg_up": weighted_mean(
                            g["avg_pages_up"], g["total_simulations"]
                        ),
                        "avg_down": weighted_mean(
                            g["avg_pages_down"], g["total_simulations"]
                        ),
                        "avg_neutral": weighted_mean(
                            g["avg_pages_neutral"], g["total_simulations"]
                        ),
                        "down_up": weighted_mean(
                            g["down_up_ratio"], g["total_simulations"]
                        ),
                    }
                )
            )
            .reset_index()
        )

        summary["strategy"] = pd.Categorical(
            summary["strategy"], categories=STRATEGY_ORDER, ordered=True
        )
        summary = summary.sort_values("strategy")

        x = np.arange(len(summary))
        colors = [
            STRATEGY_COLORS.get(s, TAB10(0)) for s in summary["strategy"].astype(str)
        ]

        # Top left: Authority yield with volatility error bars
        ax = axes[0, 0]
        y = summary["yield_mean"].to_numpy(dtype=float)
        yerr = summary["volatility_mean"].to_numpy(dtype=float)
        yerr = np.where(np.isfinite(yerr), yerr, 0.0)

        ax.bar(x, y, color=colors)
        ax.errorbar(x, y, yerr=yerr, fmt="none", capsize=4, ecolor="black", alpha=0.5)
        ax.set_title("")
        ax.set_ylabel("Authority yield")
        ax.set_xticks(x)
        ax.set_xticklabels(summary["strategy"].astype(str), rotation=30, ha="right")

        # Top right: Page distribution
        ax = axes[0, 1]
        up = summary["avg_up"].to_numpy(dtype=float)
        down = summary["avg_down"].to_numpy(dtype=float)
        neutral = summary["avg_neutral"].to_numpy(dtype=float)

        if np.all(np.isnan(up)) or np.all(np.isnan(down)):
            ax.text(
                0.5,
                0.5,
                "Page distribution data missing",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_axis_off()
        else:
            if np.all(np.isnan(neutral)):
                neutral = self.config.TOTAL_KALICUBE_PAGES - (
                    np.nan_to_num(up) + np.nan_to_num(down)
                )

            neutral = np.maximum(neutral, 0)

            ax.bar(x, up, label="Up", color=DISTRIBUTION_COLORS["up"])
            ax.bar(
                x,
                down,
                bottom=np.nan_to_num(up),
                label="Down",
                color=DISTRIBUTION_COLORS["down"],
            )
            ax.bar(
                x,
                neutral,
                bottom=np.nan_to_num(up) + np.nan_to_num(down),
                label="Neutral",
                color=DISTRIBUTION_COLORS["neutral"],
            )
            ax.set_title("")
            ax.set_ylabel("Average number of pages")
            ax.set_xticks(x)
            ax.set_xticklabels(summary["strategy"].astype(str), rotation=30, ha="right")
            ax.legend(title=None)

        # Bottom left: Down Up ratio
        ax = axes[1, 0]
        ax.bar(x, summary["down_up"].to_numpy(dtype=float), color=colors)
        ax.set_title("")
        ax.set_ylabel("Down/Up ratio")
        ax.set_xticks(x)
        ax.set_xticklabels(summary["strategy"].astype(str), rotation=30, ha="right")

        # Bottom right: Authority volatility
        ax = axes[1, 1]
        ax.bar(x, summary["volatility_mean"].to_numpy(dtype=float), color=colors)
        ax.set_title("")
        ax.set_ylabel("Authority volatility")
        ax.set_xticks(x)
        ax.set_xticklabels(summary["strategy"].astype(str), rotation=30, ha="right")

        plt.tight_layout()
        self.figures.append(fig)

    def _create_comparative_plots(self, df: pd.DataFrame):
        fig, axes = plt.subplots(2, 2, figsize=(20, 14))
        # No suptitle and no panel titles, per professor feedback

        df = df.copy()
        df["Connection range"] = df["range"].apply(normalize_range_to_label)

        range_order = sorted(
            df["Connection range"].dropna().unique().tolist(), key=range_sort_key
        )
        df["Connection range"] = pd.Categorical(
            df["Connection range"], categories=range_order, ordered=True
        )

        # Top left: Authority yield by range and type
        ax = axes[0, 0]
        if "type" in df.columns and df["type"].notna().any():
            sns.boxplot(
                data=df,
                x="Connection range",
                y="authority_yield",
                hue="type",
                hue_order=TYPE_ORDER,
                palette=TYPE_COLORS,
                ax=ax,
            )
            ax.legend(title=None)
        else:
            sns.boxplot(data=df, x="Connection range", y="authority_yield", ax=ax)

        ax.set_title("")
        ax.set_xlabel("Connection range")
        ax.set_ylabel("Authority yield")

        # Top right: Best strategy per range
        ax = axes[0, 1]
        perf = (
            df.groupby(["Connection range", "strategy"], observed=True)
            .apply(lambda g: group_weighted_mean(g, "authority_yield"))
            .reset_index(name="authority_yield")
            .dropna()
        )

        if not perf.empty:
            best_idx = perf.groupby("Connection range", observed=True)[
                "authority_yield"
            ].idxmax()
            best = perf.loc[best_idx].copy()
            best["strategy"] = pd.Categorical(
                best["strategy"], categories=STRATEGY_ORDER, ordered=True
            )

            sns.barplot(
                data=best,
                x="Connection range",
                y="authority_yield",
                hue="strategy",
                hue_order=STRATEGY_ORDER,
                palette=STRATEGY_COLORS,
                dodge=False,
                ax=ax,
            )
            ax.set_title("")
            ax.set_xlabel("Connection range")
            ax.set_ylabel("Authority yield")
            ax.legend(title=None, bbox_to_anchor=(1.05, 1), loc="upper left")
        else:
            ax.set_axis_off()

        # Bottom left: Down Up ratio by range and type
        ax = axes[1, 0]
        if "type" in df.columns and df["type"].notna().any():
            sns.violinplot(
                data=df,
                x="Connection range",
                y="down_up_ratio",
                hue="type",
                hue_order=TYPE_ORDER,
                palette=TYPE_COLORS,
                split=True,
                ax=ax,
            )
            ax.legend(title=None)
        else:
            sns.violinplot(data=df, x="Connection range", y="down_up_ratio", ax=ax)

        ax.set_title("")
        ax.set_xlabel("Connection range")
        ax.set_ylabel("Down/Up ratio")

        # Bottom right: Scatter tradeoff
        ax = axes[1, 1]
        scatter = (
            df.groupby(["strategy", "type", "Connection range"], observed=True)
            .apply(
                lambda g: pd.Series(
                    {
                        "authority_yield": group_weighted_mean(g, "authority_yield"),
                        "down_up_ratio": group_weighted_mean(g, "down_up_ratio"),
                    }
                )
            )
            .reset_index()
        )

        sns.scatterplot(
            data=scatter,
            x="authority_yield",
            y="down_up_ratio",
            hue="strategy",
            style="type" if "type" in scatter.columns else None,
            alpha=0.75,
            palette=STRATEGY_COLORS,
            hue_order=STRATEGY_ORDER,
            ax=ax,
        )

        ax.set_title("")
        ax.set_xlabel("Authority yield")
        ax.set_ylabel("Down/Up ratio")
        ax.legend(title=None, bbox_to_anchor=(1.05, 1), loc="upper left")

        plt.tight_layout()
        self.figures.append(fig)

    def save_figures(self, output_path: Optional[str] = None):
        if not output_path:
            output_path = self.config.OUTPUT_PATH

        plot_names = [
            "triad_strategy_performance",
            "triad_comparative_analysis",
        ]

        for fig, name in zip(self.figures, plot_names):
            plot_path = f"{output_path}/{name}.png"
            fig.savefig(plot_path, dpi=self.config.DPI, bbox_inches="tight")
            print(f"Saved {name} to {plot_path}")

        pdf_path = f"{output_path}/final_metric_triad_analysis.pdf"
        with PdfPages(pdf_path) as pdf:
            for fig in self.figures:
                pdf.savefig(fig, bbox_inches="tight")
        print(f"Saved PDF to {pdf_path}")


# ============================================================
# App
# ============================================================


class StrategyAnalyzerApp:
    def __init__(self):
        self.config = Config()
        self.file_manager = FileManager()
        self.visualizer = UnifiedVisualizer()

    def run(self):
        print("Starting analysis")

        all_files = self.file_manager.find_multi_files()
        if not all_files:
            print("No multi csv files found")
            return None

        master_df = self._load_and_combine_data(all_files)
        if master_df.empty:
            print("No data could be loaded")
            return None

        self._print_validation(master_df)
        self.visualizer.create_all_plots(master_df)

        if self._ask_to_save("plots"):
            self.visualizer.save_figures()

        if self._ask_to_save("results"):
            csv_path = os.path.join(self.config.OUTPUT_PATH, "master_triad_results.csv")
            master_df.to_csv(csv_path, index=False)
            print(f"Saved results to {csv_path}")

        return master_df

    def _load_and_combine_data(self, all_files: List[FileInfo]) -> pd.DataFrame:
        print("\nLoading and combining files")

        all_dfs: List[pd.DataFrame] = []

        strategy_mapping = {
            "low": "low candidates",
            "low candidates": "low candidates",
            "worst": "low candidates",
            "worst candidates": "low candidates",
            "high": "high candidates",
            "high candidates": "high candidates",
            "best": "high candidates",
            "best candidates": "high candidates",
            "mixed": "mixed candidates",
            "mixed candidates": "mixed candidates",
            "random": "random candidates",
            "random candidates": "random candidates",
            "folder": "folder candidates",
            "folder candidates": "folder candidates",
        }

        default_k = float(self.config.INSERTED_LINKS_K)

        for file_info in all_files:
            try:
                df = pd.read_csv(file_info.full_path)
                df["type"] = file_info.type.value
                df["sub_category"] = file_info.sub_category.value
                df["source_file"] = file_info.filename

                # Strategy normalization
                if "strategy" in df.columns:
                    df["strategy"] = df["strategy"].astype(str).str.lower().str.strip()
                    df["strategy"] = df["strategy"].replace(strategy_mapping)

                # Range normalization
                if "range" in df.columns:
                    df["range"] = df["range"].astype(str).str.strip()
                else:
                    df["range"] = "unknown"

                # Total simulations weight
                if "total_simulations" in df.columns:
                    df["total_simulations"] = ensure_numeric(
                        df["total_simulations"]
                    ).fillna(1.0)
                else:
                    df["total_simulations"] = 1.0

                # k used
                if "k_used" in df.columns:
                    df["k_used"] = ensure_numeric(df["k_used"]).fillna(default_k)
                else:
                    df["k_used"] = default_k

                # Authority Yield per inserted link
                mean_col = pick_first_existing_col(df, MEAN_COL_CANDIDATES)
                raw_mean = (
                    ensure_numeric(df[mean_col]) if mean_col is not None else None
                )
                df["authority_yield"] = (
                    (raw_mean / df["k_used"]) if raw_mean is not None else np.nan
                )

                # Authority Volatility per inserted link
                std_col = pick_first_existing_col(df, STD_COL_CANDIDATES)
                if std_col is not None:
                    raw_std = ensure_numeric(df[std_col])
                    df["authority_volatility"] = raw_std / df["k_used"]
                else:
                    print("\nWARNING missing std delta column")
                    print(f"File {file_info.relative_path}")
                    print("Volatility will be NaN for rows from this file")
                    print(f"Available columns {df.columns.tolist()}\n")
                    df["authority_volatility"] = np.nan

                # Down Up Ratio
                if "avg_pages_down" in df.columns and "avg_pages_up" in df.columns:
                    down = ensure_numeric(df["avg_pages_down"])
                    up = ensure_numeric(df["avg_pages_up"])
                    df["down_up_ratio"] = down / np.maximum(up, 1.0)
                else:
                    df["down_up_ratio"] = np.nan

                # Neutral pages context
                if "avg_pages_neutral" in df.columns:
                    df["avg_pages_neutral"] = ensure_numeric(df["avg_pages_neutral"])
                elif "avg_pages_up" in df.columns and "avg_pages_down" in df.columns:
                    up = ensure_numeric(df["avg_pages_up"])
                    down = ensure_numeric(df["avg_pages_down"])
                    neutral = self.config.TOTAL_KALICUBE_PAGES - (up + down)
                    df["avg_pages_neutral"] = np.maximum(neutral, 0)
                else:
                    df["avg_pages_neutral"] = np.nan

                all_dfs.append(df)

            except Exception as e:
                print(f"Could not process {file_info.filename}: {e}")

        if not all_dfs:
            return pd.DataFrame()

        master_df = pd.concat(all_dfs, ignore_index=True)

        # Categorical ordering, keep fixed strategy order
        if "strategy" in master_df.columns:
            unknown_mask = (
                ~master_df["strategy"].isin(STRATEGY_ORDER)
                & master_df["strategy"].notna()
            )
            if unknown_mask.any():
                print("\nWARNING dropping rows with unknown strategies")
                print(master_df.loc[unknown_mask, "strategy"].value_counts())
                master_df = master_df.loc[~unknown_mask].copy()

            master_df["strategy"] = pd.Categorical(
                master_df["strategy"], categories=STRATEGY_ORDER, ordered=True
            )

        if "type" in master_df.columns:
            master_df["type"] = pd.Categorical(
                master_df["type"], categories=TYPE_ORDER, ordered=True
            )

        print(f"Combined {len(all_dfs)} files into {len(master_df)} rows")
        return master_df

    def _print_validation(self, df: pd.DataFrame):
        print("\nData validation")
        print("=" * 60)

        print("Columns present")
        print(df.columns.tolist())

        if "strategy" in df.columns:
            print("\nStrategies found")
            print(df["strategy"].dropna().unique().tolist())

        cols = ["authority_yield", "authority_volatility", "down_up_ratio"]
        existing = [c for c in cols if c in df.columns]
        if existing and "strategy" in df.columns:
            stats = df.groupby("strategy", observed=True)[existing].mean()
            print("\nMean metrics per strategy")
            print(stats)

        if "authority_yield" in df.columns:
            print(f"\nMissing yield {df['authority_yield'].isna().mean():.2%}")
        if "authority_volatility" in df.columns:
            print(f"Missing volatility {df['authority_volatility'].isna().mean():.2%}")

        print("=" * 60 + "\n")

    def _ask_to_save(self, item: str) -> bool:
        response = input(f"\nSave {item} to Drive (y or n): ").strip().lower()
        return response == "y"


def main():
    app = StrategyAnalyzerApp()
    results_df = app.run()

    if results_df is not None and not results_df.empty:
        print("\n" + "=" * 60)
        print("Analysis complete")
        print("=" * 60)

    return results_df


if __name__ == "__main__":
    master_results = main()